# Evaluate ResNet-50 on ISIC 2019 (Test Set)

This notebook reloads a **trained ResNet-50 checkpoint** and evaluates it on the **test split** (either the official ISIC test set with ground-truth, or the internal train/val/test split used in `ISICDataset`).

Outputs:
- Metrics: accuracy, macro-F1, per-class report, confusion matrix
- Saved files: `../results/resnet50_test_metrics.json`, `../results/resnet50_test_predictions.csv`, `../results/resnet50_test_confusion_matrix.png`

In [ ]:
# ==============================
# 0) Imports
# ==============================
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

# Make project imports work from /notebooks
sys.path.insert(0, str(Path('..').resolve()))

from data.isic_dataset import ISICDataset, CLASS_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ==============================
# 1) Paths / Settings
# ==============================
# Dataset root directory should contain either:
# - (official test) ISIC_2019_Test_Input/ + ISIC_2019_Test_GroundTruth.csv
# - (train-only split) ISIC_2019_Training_Input/ + ISIC_2019_Training_GroundTruth.csv

# Local default (repo structure)
ISIC_ROOT = Path('../data/ISIC2019').resolve()

# Kaggle checkpoint path (your path) — used if it exists
KAGGLE_CKPT = Path('/kaggle/input/models/youssefnouiouar1/resnet50/pytorch/default/1/resnet50_best.pth')

# Local checkpoint fallback (repo already has it) — used if it exists
LOCAL_CKPT = Path('../results/resnet50_best.pth').resolve()

# Choose which test definition to use:
# - True  => official ISIC test set (requires test GT CSV)
# - False => internal split made from training CSV (deterministic via random_state)
USE_OFFICIAL_TEST = True

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
RANDOM_STATE = 42
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# Output files
OUT_DIR = Path('../results').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = OUT_DIR / 'resnet50_test_metrics.json'
PRED_CSV_PATH = OUT_DIR / 'resnet50_test_predictions.csv'
CM_PNG_PATH = OUT_DIR / 'resnet50_test_confusion_matrix.png'

# Pick checkpoint path
if KAGGLE_CKPT.exists():
    CKPT_PATH = KAGGLE_CKPT
elif LOCAL_CKPT.exists():
    CKPT_PATH = LOCAL_CKPT
else:
    CKPT_PATH = KAGGLE_CKPT  # keep your path as default

print('ISIC_ROOT:', ISIC_ROOT)
print('Checkpoint:', CKPT_PATH)
print('USE_OFFICIAL_TEST:', USE_OFFICIAL_TEST)

In [ ]:
# ==============================
# 2) DataLoader (Test)
# ==============================
test_ds = ISICDataset(
    root_dir=str(ISIC_ROOT),
    split='test',
    image_size=IMAGE_SIZE,
    use_albumentations=True,
    use_official_test=USE_OFFICIAL_TEST,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    random_state=RANDOM_STATE,
    augmentation_strength='medium'
)

num_classes = len(test_ds.class_names)
print('Test samples:', len(test_ds))
print('Num classes :', num_classes)
print('Classes     :', test_ds.class_names)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
# ==============================
# 3) Build model + load checkpoint
# ==============================
def build_resnet50(num_classes: int):
    model = timm.create_model('resnet50', pretrained=False, num_classes=num_classes)
    return model

def extract_state_dict(ckpt_obj):
    # Common patterns in this repo / notebooks:
    # - raw state_dict
    # - {'model_state_dict': state_dict, ...}
    # - {'state_dict': state_dict, ...}
    # - {'model': state_dict, ...}
    if isinstance(ckpt_obj, dict):
        for key in ['model_state_dict', 'state_dict', 'model']:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                return ckpt_obj[key]
    if isinstance(ckpt_obj, dict):
        # If it's already a param dict
        if all(isinstance(k, str) for k in ckpt_obj.keys()):
            return ckpt_obj
    raise ValueError('Unrecognized checkpoint format')

def clean_state_dict(state_dict):
    # Remove DataParallel prefix if present
    return {k.replace('module.', ''): v for k, v in state_dict.items()}

model = build_resnet50(num_classes=num_classes).to(device)
model.eval()

if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
state = clean_state_dict(extract_state_dict(ckpt))

missing, unexpected = model.load_state_dict(state, strict=False)
print('Loaded checkpoint:', CKPT_PATH)
print('Missing keys   :', len(missing))
print('Unexpected keys:', len(unexpected))

# Some checkpoints are saved from a wrapper and contain keys like 'model.xxx'
if len(unexpected) > 0 and any(k.startswith('model.') for k in state.keys()):
    # Retry: strip leading 'model.'
    stripped = {k.replace('model.', '', 1): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(stripped, strict=False)
    print('Retried with stripped prefix "model."')
    print('Missing keys   :', len(missing))
    print('Unexpected keys:', len(unexpected))

In [ ]:
# ==============================
# 4) Evaluation loop
# ==============================
@torch.no_grad()
def evaluate(model, loader):
    y_true = []
    y_pred = []
    y_prob = []
    img_names = []

    for images, labels, names in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.append(labels.cpu().numpy())
        y_pred.append(preds.cpu().numpy())
        y_prob.append(probs.cpu().numpy())
        img_names.extend(list(names))

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_prob = np.concatenate(y_prob)
    return y_true, y_pred, y_prob, img_names

y_true, y_pred, y_prob, img_names = evaluate(model, test_loader)
print('Done. N =', len(y_true))

In [ ]:
# ==============================
# 5) Metrics + confusion matrix
# ==============================
acc = float(accuracy_score(y_true, y_pred))
f1_macro = float(f1_score(y_true, y_pred, average='macro'))
f1_weighted = float(f1_score(y_true, y_pred, average='weighted'))

report = classification_report(
    y_true, y_pred, target_names=test_ds.class_names, output_dict=True, zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

# Multiclass AUC (OVR). Can fail if a class is absent in y_true.
auc_ovr_macro = None
try:
    y_true_onehot = np.eye(num_classes)[y_true]
    auc_ovr_macro = float(roc_auc_score(y_true_onehot, y_prob, multi_class='ovr', average='macro'))
except Exception as e:
    print('AUC not computed:', e)
metrics = {
    'checkpoint': str(CKPT_PATH),
    'isic_root': str(ISIC_ROOT),
    'use_official_test': bool(USE_OFFICIAL_TEST),
    'n_test': int(len(y_true)),
    'num_classes': int(num_classes),
    'class_names': list(test_ds.class_names),
    'accuracy': acc,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'auc_ovr_macro': auc_ovr_macro,
    'classification_report': report,
}
print('accuracy     :', acc)
print('f1_macro     :', f1_macro)
print('f1_weighted  :', f1_weighted)
print('auc_ovr_macro:', auc_ovr_macro)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_ds.class_names, yticklabels=test_ds.class_names)
plt.title('ResNet-50 — Confusion Matrix (Test)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 6) Save outputs (metrics + predictions)
# ==============================
# Save metrics JSON
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Saved:', METRICS_PATH)

# Save confusion matrix figure
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_ds.class_names, yticklabels=test_ds.class_names)
plt.title('ResNet-50 — Confusion Matrix (Test)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(CM_PNG_PATH, dpi=200)
plt.close()
print('Saved:', CM_PNG_PATH)

# Save per-image predictions
pred_df = pd.DataFrame({
    'image': img_names,
    'y_true': y_true,
    'y_pred': y_pred,
    'true_class': [test_ds.class_names[i] for i in y_true],
    'pred_class': [test_ds.class_names[i] for i in y_pred],
})
for i, cls in enumerate(test_ds.class_names):
    pred_df[f'prob_{cls}'] = y_prob[:, i]
pred_df.to_csv(PRED_CSV_PATH, index=False)
print('Saved:', PRED_CSV_PATH)

# Quick peek
pred_df.head()